In [ ]:
import sys
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

# 从当前目录向上找 .git，定位根目录
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".git").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
# 把项目根目录加入到sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DB_PATH, FIGURES_DIR, ensure_dirs

ensure_dirs()

# 设置中文字体以及负号显示
from src.viz import apply_chinese, get_font_prop

font_prop = get_font_prop()
plt.rcParams["axes.unicode_minus"] = False


con = duckdb.connect(str(DB_PATH), read_only=True)


In [ ]:
retention = con.sql("""

WITH daily_active AS (
    SELECT DISTINCT user_id, DATE_TRUNC('day',time) AS dt
    FROM user_behavior
    ORDER BY user_id,dt
),
first_date AS (
SELECT user_id, MIN(DATE_TRUNC('day',time)) AS first_dt
FROM user_behavior
GROUP BY user_id
)

SELECT 
    f.first_dt,
    COUNT(DISTINCT f.user_id) AS new_users,
    COUNT(DISTINCT CASE WHEN DATE_DIFF('day', f.first_dt, d.dt) = 1 THEN d.user_id END) AS day1_users,
    COUNT(DISTINCT CASE WHEN DATE_DIFF('day', f.first_dt, d.dt) = 7 THEN d.user_id END) AS day7_users
FROM  first_date f LEFT JOIN daily_active d
ON f.user_id = d.user_id
GROUP BY f.first_dt
ORDER BY f.first_dt
""").df()
retention['day1_rate'] = round(retention['day1_users']/retention['new_users'] * 100,2)
retention['day7_rate'] = round(retention['day7_users']/retention['new_users'] * 100,2)
retention

In [ ]:
# 留存曲线
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(retention['first_dt'], retention['day1_rate'],marker="o", label="次日留存",color="#4C72B0")
ax.plot(retention['first_dt'], retention['day7_rate'],marker="o", label="7日留存",color="#C44E52")
ax.set_title("留存趋势")
ax.set_xlabel("首日日期")
ax.set_ylabel("留存率(%)")
ax.legend()
ax.tick_params(axis="x", rotation=45)
apply_chinese(ax,font_prop)

plt.savefig(FIGURES_DIR/"retention_curve.png")
plt.show()
